In [0]:

titles_path = "abfss://bronze@netflixxdata.dfs.core.windows.net/netflix_titles"

df_titles = spark.read.format("delta").load(titles_path)

print("Netflix Titles records:", df_titles.count())


Netflix Titles records: 6236


In [0]:

from pyspark.sql.functions import col, trim

source = "abfss://bronze@netflixxdata.dfs.core.windows.net/netflix_titles"
target = "abfss://silver@netflixxdata.dfs.core.windows.net/netflix_titles"

# Read Bronze Delta data
df = spark.read.format("delta").load(source)

# Apply basic Silver cleaning
df_clean = (
    df.filter(col("show_id").isNotNull())
      .dropDuplicates(["show_id"])
      .withColumn("title", trim(col("title")))
      .withColumn("release_year", col("release_year").cast("int"))
)

# Write cleaned data to Silver
(
    df_clean.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .save(target)
)

print("Netflix Titles successfully loaded into Silver")


Netflix Titles successfully loaded into Silver
